# Audit Hypothesis Agent — Multi-Agent (LangGraph)

Расширение `rag_system_v2.ipynb`: добавляет мультиагентный граф с циклом самопроверки.

## Архитектура графа

```
START
  │
  ▼
[Planner]      — декомпозирует вопрос на 2-3 фокусированных под-вопроса
  │
  ▼
[Retriever]    — параллельный поиск по всем под-вопросам + reranking
  │
  ▼
[Generator]    — генерирует первичные гипотезы
  │
  ▼
[Critic]       — оценивает качество (0-10), выявляет пробелы
  │
  ├─ score < 7 и iterations < 2 ──► [Refiner] ──► [Critic]  (цикл)
  │
  └─ score ≥ 7 или max_iter ──────► [Reporter]
                                        │
                                       END
```

## Что добавляет каждый агент

| Агент | LLM-вызов | Что делает |
|-------|-----------|------------|
| Planner | ~150 токенов | Разбивает вопрос на под-темы → лучший recall |
| Retriever | нет | Поиск + reranking по всем под-темам |
| Generator | ~2000 токенов | Первичные гипотезы |
| Critic | ~600 токенов | Оценка 0-10 + конкретные замечания |
| Refiner | ~1500 токенов | Улучшение по замечаниям Critic |
| Reporter | нет | Форматирование в HypothesesReport |

**Итого:** 3-5 LLM-вызовов против 2 в v2. На GPU (4-bit) — приемлемо. На CPU — медленно.

## 1. Установка LangGraph

In [ ]:
%pip install -q langgraph langchain-core

## 2. Запуск rag_system_v2

Перед этим ноутбуком должен быть выполнен `rag_system_v2.ipynb`.
Все классы (DocumentRetriever, AuditHypothesis, AuditAgent и т.д.) переиспользуются.

In [ ]:
# Проверяем, что нужные объекты загружены из rag_system_v2
assert 'agent' in dir(), "Запусти сначала все ячейки rag_system_v2.ipynb"
assert 'model' in dir() and 'tokenizer' in dir()

from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict, Optional, Annotated
import operator

print("LangGraph готов к работе")

## 3. State — общее состояние графа

In [ ]:
class AuditState(TypedDict):
    """
    Состояние, передаваемое между всеми агентами графа.
    Annotated[List, operator.add] означает, что узлы могут только добавлять элементы.
    """
    # Входные данные
    question: str
    max_iterations: int

    # Плanner
    sub_questions: List[str]

    # Retriever
    chunks: List[Dict]

    # Generator / Refiner
    raw_hypotheses: str

    # Critic
    critique: str
    quality_score: int       # 0-10, порог для рефайна = 7
    iterations: int          # счётчик итераций Critic→Refiner

    # Reporter
    final_report: Optional[Dict]  # сериализованный HypothesesReport.dict()

## 4. Вспомогательная функция генерации

In [ ]:
def llm_call(system: str, user: str,
             max_new_tokens: int = 512,
             temperature: float = 0.4) -> str:
    """Единая точка вызова LLM для всех агентов графа."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    ids = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(ids, skip_special_tokens=True).strip()

## 5. Узлы графа (агенты)

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# PLANNER
# Декомпозирует исходный вопрос на 2-3 фокусированных под-вопроса.
# Цель: расширить охват retrieval без лишних запросов.
# ─────────────────────────────────────────────────────────────────────
def planner_node(state: AuditState) -> dict:
    print("[Planner] Декомпозиция вопроса...")
    raw = llm_call(
        system="Ты — опытный аудитор банковской группы.",
        user=(
            f"Разбей следующий аудиторский вопрос на 2-3 узкие под-темы "
            f"для более точного поиска по документам.\n"
            f"Каждую под-тему напиши на отдельной строке. Без нумерации и пояснений.\n\n"
            f"Вопрос: {state['question']}"
        ),
        max_new_tokens=200,
        temperature=0.3,
    )
    sub_questions = [state["question"]] + [
        line.strip() for line in raw.splitlines() if line.strip()
    ][:3]
    print(f"  Под-вопросов: {len(sub_questions)}")
    for q in sub_questions[1:]:
        print(f"  • {q}")
    return {"sub_questions": sub_questions}


# ─────────────────────────────────────────────────────────────────────
# RETRIEVER
# Ищет по всем под-вопросам, дедуплицирует, ранжирует через CrossEncoder.
# LLM не вызывается — только векторный поиск.
# ─────────────────────────────────────────────────────────────────────
def retriever_node(state: AuditState) -> dict:
    print("[Retriever] Поиск по всем под-вопросам...")
    all_results = []
    for q in state["sub_questions"]:
        all_results.extend(agent.retriever.search(q, top_k=6))

    # Дедупликация по тексту
    unique = list({r["text"]: r for r in all_results}.values())

    if not unique:
        print("  ⚠ База знаний пуста")
        return {"chunks": []}

    # Reranking по исходному вопросу
    pairs = [[state["question"], r["text"]] for r in unique]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(unique, scores), key=lambda x: x[1], reverse=True)
    chunks = [r for r, _ in ranked[:8]]
    print(f"  Отобрано фрагментов: {len(chunks)}")
    return {"chunks": chunks}


# ─────────────────────────────────────────────────────────────────────
# GENERATOR
# Генерирует первичные гипотезы на основе контекста.
# ─────────────────────────────────────────────────────────────────────
def generator_node(state: AuditState) -> dict:
    print("[Generator] Генерация гипотез...")
    context = agent._build_context(state["chunks"])
    prompt = agent._build_prompt(context, state["question"])
    raw = llm_call(
        system="Ты — ассистент ведущего аудитора банковской группы.",
        user=prompt,
        max_new_tokens=2048,
        temperature=0.6,
    )
    print(f"  Сгенерировано {len(raw)} символов")
    return {"raw_hypotheses": raw, "iterations": 0}


# ─────────────────────────────────────────────────────────────────────
# CRITIC
# Оценивает качество гипотез по 3 критериям:
#   1. Конкретность (конкретные ли риски, не общие фразы)
#   2. Проверяемость (есть ли конкретные шаги проверки)
#   3. Полнота (охвачены ли ключевые риски из контекста)
# Возвращает оценку 0-10 и конкретные замечания.
# ─────────────────────────────────────────────────────────────────────
def critic_node(state: AuditState) -> dict:
    iteration = state.get("iterations", 0) + 1
    print(f"[Critic] Оценка качества (итерация {iteration})...")

    raw = llm_call(
        system="Ты — строгий эксперт по аудиту. Оцениваешь качество аудиторских гипотез.",
        user=(
            f"Оцени следующие аудиторские гипотезы по трём критериям:\n"
            f"1. Конкретность: указаны ли конкретные риски (не общие фразы)?\n"
            f"2. Проверяемость: есть ли конкретные шаги проверки?\n"
            f"3. Полнота: охвачены ли ключевые риски из контекста?\n\n"
            f"Формат ответа:\n"
            f"ОЦЕНКА: [число от 0 до 10]\n"
            f"ЗАМЕЧАНИЯ:\n"
            f"- [замечание 1]\n"
            f"- [замечание 2]\n\n"
            f"Исходный вопрос: {state['question']}\n\n"
            f"Гипотезы:\n{state['raw_hypotheses'][:3000]}"
        ),
        max_new_tokens=600,
        temperature=0.3,
    )

    # Извлекаем числовую оценку
    score_match = re.search(r"ОЦЕНКА:\s*(\d+)", raw)
    quality_score = int(score_match.group(1)) if score_match else 5
    quality_score = max(0, min(10, quality_score))  # clamp 0-10

    print(f"  Оценка: {quality_score}/10")
    if quality_score < 7:
        print("  → Требуется доработка")
    else:
        print("  → Качество достаточное")

    return {"critique": raw, "quality_score": quality_score, "iterations": iteration}


# ─────────────────────────────────────────────────────────────────────
# REFINER
# Дорабатывает гипотезы на основе замечаний Critic.
# Получает исходные гипотезы + критику + контекст.
# ─────────────────────────────────────────────────────────────────────
def refiner_node(state: AuditState) -> dict:
    print(f"[Refiner] Доработка гипотез...")
    context = agent._build_context(state["chunks"])
    raw = llm_call(
        system="Ты — ассистент ведущего аудитора банковской группы.",
        user=(
            f"Улучши аудиторские гипотезы на основе следующих замечаний эксперта.\n\n"
            f"ЗАМЕЧАНИЯ ЭКСПЕРТА:\n{state['critique']}\n\n"
            f"ИСХОДНЫЕ ГИПОТЕЗЫ:\n{state['raw_hypotheses'][:2000]}\n\n"
            f"КОНТЕКСТ ДОКУМЕНТОВ:\n{context[:4000]}\n\n"
            f"Исходный вопрос: {state['question']}\n\n"
            f"Требования:\n"
            f"- Сохрани хорошие гипотезы, улучши слабые\n"
            f"- Конкретизируй расплывчатые формулировки\n"
            f"- Добавь конкретные шаги проверки там, где их нет\n"
            f"- Укажи уровень риска: HIGH / MEDIUM / LOW"
        ),
        max_new_tokens=2048,
        temperature=0.5,
    )
    print(f"  Доработано: {len(raw)} символов")
    return {"raw_hypotheses": raw}


# ─────────────────────────────────────────────────────────────────────
# REPORTER
# Парсит финальный текст в структурированный HypothesesReport.
# LLM не вызывается.
# ─────────────────────────────────────────────────────────────────────
def reporter_node(state: AuditState) -> dict:
    print("[Reporter] Формирование отчёта...")
    report = parse_hypotheses(
        state["raw_hypotheses"],
        state["question"],
        state["chunks"],
    )
    print(f"  Гипотез: {len(report.hypotheses)}, источников: {report.sources_used}")
    return {"final_report": report.dict()}

## 6. Условное ребро: рефайн или финиш?

In [ ]:
def should_refine_or_report(state: AuditState) -> str:
    """
    Маршрутизатор после Critic.
    Переходим к Refiner если:
      - оценка ниже порога (< 7)
      - ещё не исчерпан лимит итераций
    Иначе — сразу к Reporter.
    """
    score      = state.get("quality_score", 0)
    iterations = state.get("iterations", 0)
    max_iter   = state.get("max_iterations", 2)

    if score < 7 and iterations < max_iter:
        print(f"  ↻ Оценка {score}/10 < 7, итерация {iterations}/{max_iter} → Refiner")
        return "refine"
    else:
        print(f"  ✓ Оценка {score}/10, итерация {iterations}/{max_iter} → Reporter")
        return "report"

## 7. Сборка графа

In [ ]:
# Собираем StateGraph
workflow = StateGraph(AuditState)

# Регистрируем узлы
workflow.add_node("planner",   planner_node)
workflow.add_node("retriever", retriever_node)
workflow.add_node("generator", generator_node)
workflow.add_node("critic",    critic_node)
workflow.add_node("refiner",   refiner_node)
workflow.add_node("reporter",  reporter_node)

# Линейные рёбра
workflow.set_entry_point("planner")
workflow.add_edge("planner",   "retriever")
workflow.add_edge("retriever", "generator")
workflow.add_edge("generator", "critic")
workflow.add_edge("refiner",   "critic")   # цикл: Refiner → Critic
workflow.add_edge("reporter",  END)

# Условное ребро после Critic
workflow.add_conditional_edges(
    "critic",
    should_refine_or_report,
    {"refine": "refiner", "report": "reporter"},
)

# Компилируем граф
audit_graph = workflow.compile()
print("Граф скомпилирован ✓")

# Визуализация (если установлен graphviz)
try:
    from IPython.display import Image, display
    display(Image(audit_graph.get_graph().draw_mermaid_png()))
except Exception:
    print("Установи graphviz для визуализации: pip install graphviz")

## 8. Запуск

In [ ]:
question = (
    "Вычитка информации о финансовых транзакциях по бизнес-карте "
    "корпоративного клиента и их отображение по счёту клиента"
)

initial_state: AuditState = {
    "question":       question,
    "max_iterations": 2,          # максимум 2 цикла Critic→Refiner
    "sub_questions":  [],
    "chunks":         [],
    "raw_hypotheses": "",
    "critique":       "",
    "quality_score":  0,
    "iterations":     0,
    "final_report":   None,
}

print("=" * 60)
print(f"Вопрос: {question}")
print("=" * 60)

final_state = audit_graph.invoke(initial_state)

## 9. Вывод результатов

In [ ]:
# Восстанавливаем HypothesesReport из сериализованного dict
report = HypothesesReport(**final_state["final_report"])

print("\n" + "=" * 60)
print(report.to_markdown())

print("\n" + "=" * 60)
print(f"Итоговая оценка Critic : {final_state['quality_score']}/10")
print(f"Итераций Critic→Refiner: {final_state['iterations']}")
print(f"Под-вопросов (Planner) : {len(final_state['sub_questions'])}")
print(f"Фрагментов использовано: {report.sources_used}")
if report.source_names:
    print("Источники:")
    for s in report.source_names:
        print(f"  • {s}")

## 10. Потоковый вывод (stream)

LangGraph позволяет смотреть результат каждого агента по мере выполнения.

In [ ]:
# Альтернативный запуск с потоковым выводом:
# каждый шаг графа выводится сразу после завершения

print("Запуск с потоковым выводом...\n")

for step in audit_graph.stream(initial_state):
    node_name = list(step.keys())[0]
    node_output = step[node_name]
    print(f"\n── {node_name.upper()} ──")
    # Показываем ключевые поля шага
    for key, val in node_output.items():
        if key == "raw_hypotheses":
            print(f"  raw_hypotheses: {str(val)[:200]}...")
        elif key == "chunks":
            print(f"  chunks: {len(val)} фрагментов")
        elif key == "final_report":
            print(f"  final_report: {len(val.get('hypotheses', []))} гипотез")
        else:
            print(f"  {key}: {val}")